# RESP電荷のための波動関数

[antechamber の手順](https://qiita.com/Ag_smith/items/430e9efb32a855d4c511)に合わせ、計算を2段に分ける。

1. MACE で粗く最適化した座標を初期構造にし、**B3LYP/6-31+G(d,p)** で構造を取り直す。収束は Gaussian の `opt=tight` に相当する。
2. その座標を固定し、**Hartree–Fock/6-31G(d)** の単点計算をする。この波動関数を Multiwfn に渡し、RESP電荷をフィットする。

最初に読み込む座標の既定値は `output_mace/TAPC/TAPC_opt.xyz`。別の材料にするときは、このパスだけ変える。MACE の xyz は上書きしない。B3LYP で取り直した座標は `{名前}_resp_opt.xyz` に書く。

AMBER でよく使う RESP は、気相の HF/6-31G(d) の静電ポテンシャルに合わせた原子電荷である。Gaussian の `pop=mk iop(6/33=2,6/42=6)` は、その静電ポテンシャル格子をログへ書き出す指定にすぎない。Multiwfn は fchk の軌道から格子を自分で作るので、この指定は要らない。

出力は座標と同じフォルダに置く。

- `{名前}_resp_geomopt.log` : B3LYP 構造最適化のログ
- `{名前}_resp_opt.xyz` : B3LYP/6-31+G(d,p) の最適化構造
- `{名前}_resp.log` : HF 単点計算のログ
- `{名前}_resp.chk` : PySCFのチェックポイント
- `{名前}_resp.fchk` : Multiwfn に渡す波動関数

fchkが完成したら、Multiwfnで読み込み、    
`7（Population analysis and calculation of atomic charges）`→     
`18（Restrained ElectroStatic Potential atomic charge）`→     
`1（Start standard two-stage RESP fitting calculation）`    
で計算が始まり、chgが出力される。


In [ ]:
import os
import sys
import gc
import numpy as np
import cupy

# 座標と基底関数から分子を作る
from pyscf import gto
# 構造最適化。収束したかどうかも受け取る
from pyscf.geomopt.geometric_solver import kernel as geom_kernel
# 1段目。B3LYP の構造最適化
from gpu4pyscf.dft import rks
# 2段目。RESPに渡す波動関数は Hartree-Fock
from gpu4pyscf.scf import RHF

In [ ]:
def release_gpu():
    """計算オブジェクトを捨てたあと、CuPyが確保したままのVRAMを返す。"""
    gc.collect()
    cupy.get_default_memory_pool().free_all_blocks()
    cupy.get_default_pinned_memory_pool().free_all_blocks()

def close_pyscf_log(obj):
    """PySCFはログを開きっぱなしにする。次の計算で同じパスを使うと中身が消える。"""
    stdout = getattr(obj, "stdout", None)
    if stdout is None or stdout in (sys.stdout, sys.stderr):
        return
    try:
        stdout.flush()
    except Exception:
        pass
    try:
        if not stdout.closed:
            stdout.close()
    except Exception:
        pass

def write_fchk(mf, filename, title="PySCF SCF"):
    """収束したSCFからGaussian形式のfchkを書く。

    Multiwfnは基底関数の並びがGaussianと同じであることを前提に軌道を読む。
    PySCFのd/f関数の並びはそれと違うので、Molden出力と同じ順へ入れ替えてから書く。
    """
    from pyscf.tools.molden import order_ao_index

    def write_ints(handle, values):
        # fchkの整数配列は1行に6個
        for i, value in enumerate(values, 1):
            handle.write(f"{int(value):12d}")
            if i % 6 == 0:
                handle.write("\n")
        if len(values) % 6:
            handle.write("\n")

    def write_floats(handle, values):
        # fchkの実数配列は1行に5個
        flat = np.asarray(values, dtype=float).ravel()
        for i, value in enumerate(flat, 1):
            handle.write(f"{value:16.8E}")
            if i % 5 == 0:
                handle.write("\n")
        if len(flat) % 5:
            handle.write("\n")

    def section_array(handle, name, kind, values):
        values = list(values)
        handle.write(f"{name:<43}{kind}   N={len(values):12d}\n")
        (write_ints if kind == "I" else write_floats)(handle, values)

    def section_scalar(handle, name, kind, value):
        if kind == "I":
            handle.write(f"{name:<43}I {int(value):17d}\n")
        else:
            handle.write(f"{name:<43}R {float(value):22.15E}\n")

    mol = mf.mol
    # GPU上の軌道をCPUへコピーしてからファイルへ書く
    mo_coeff = mf.mo_coeff.get() if hasattr(mf.mo_coeff, "get") else np.asarray(mf.mo_coeff)
    mo_energy = mf.mo_energy.get() if hasattr(mf.mo_energy, "get") else np.asarray(mf.mo_energy)
    mo_coeff = mo_coeff[order_ao_index(mol)]
    nbf, nmo = mo_coeff.shape
    n_alpha, n_beta = mol.nelec

    # 各縮約殻の角運動量、原始関数、原子への対応。fchkはこれを軌道係数と一緒に持つ
    shell_type, prim_per, shell_atom = [], [], []
    exps, coeffs, shell_xyz = [], [], []
    nbf_check = 0
    for ib in range(mol.nbas):
        l = mol.bas_angular(ib)
        nctr = mol.bas_nctr(ib)
        es = mol.bas_exp(ib)
        cs = mol.bas_ctr_coeff(ib)
        xyz = mol.atom_coord(mol.bas_atom(ib))
        nfunc = (l + 1) * (l + 2) // 2 if mol.cart else 2 * l + 1
        for ic in range(nctr):
            keep = np.flatnonzero(np.abs(cs[:, ic]) > 0)
            if keep.size == 0:
                continue
            # 球面調和のd/fは負の角運動量で表す。sとpはそのまま
            shell_type.append(l if mol.cart or l <= 1 else -l)
            prim_per.append(int(keep.size))
            shell_atom.append(mol.bas_atom(ib) + 1)
            exps.extend(es[keep])
            coeffs.extend(cs[keep, ic])
            shell_xyz.extend(xyz)
            nbf_check += nfunc
    if nbf_check != nbf:
        raise RuntimeError(f"基底関数の数が一致しません: shells={nbf_check}, nao={nbf}")

    xc = getattr(mf, "xc", "HF")
    basis = mol.basis if isinstance(mol.basis, str) else "Gen"
    with open(filename, "w") as handle:
        handle.write(title + "\n")
        handle.write(f"SCF       R{str(xc).upper():<16}{basis}\n")
        section_scalar(handle, "Number of atoms", "I", mol.natm)
        section_scalar(handle, "Charge", "I", mol.charge)
        section_scalar(handle, "Multiplicity", "I", mol.spin + 1)
        section_scalar(handle, "Number of electrons", "I", mol.nelectron)
        section_scalar(handle, "Number of alpha electrons", "I", n_alpha)
        section_scalar(handle, "Number of beta electrons", "I", n_beta)
        section_scalar(handle, "Number of basis functions", "I", nbf)
        section_scalar(handle, "Number of independent functions", "I", nmo)
        section_array(handle, "Atomic numbers", "I", [mol.atom_charge(i) for i in range(mol.natm)])
        section_array(handle, "Nuclear charges", "R", mol.atom_charges())
        section_array(handle, "Current cartesian coordinates", "R", mol.atom_coords().ravel())
        section_scalar(handle, "Number of contracted shells", "I", len(shell_type))
        section_scalar(handle, "Number of primitive shells", "I", len(exps))
        section_array(handle, "Shell types", "I", shell_type)
        section_array(handle, "Number of primitives per shell", "I", prim_per)
        section_array(handle, "Shell to atom map", "I", shell_atom)
        section_array(handle, "Primitive exponents", "R", exps)
        section_array(handle, "Contraction coefficients", "R", coeffs)
        section_array(handle, "Coordinates of each shell", "R", shell_xyz)
        section_array(handle, "Alpha Orbital Energies", "R", mo_energy)
        # 列優先。1本の軌道の係数が並んでから、次の軌道に進む
        section_array(handle, "Alpha MO coefficients", "R", mo_coeff.ravel(order="F"))
        section_scalar(handle, "Total Energy", "R", mf.e_tot)

In [ ]:
# MACEで粗く最適化した座標。このファイル自体は上書きしない
xyz_path = "output_mace/TAPC/TAPC_opt.xyz"

# 1段目。構造だけを B3LYP/6-31+G(d,p) で取り直す
OPT_XC = "b3lyp"
OPT_BASIS = "6-31+g(d,p)"
# 2段目。取り直した座標を固定し、RESP用の波動関数を作る
SP_XC = "hf"
SP_BASIS = "6-31g(d)"
# xyzには電荷とスピンが入っていない。中性の閉殻なら 0 と 0
CHARGE = 0
SPIN = 0

stem = os.path.splitext(os.path.basename(xyz_path))[0]
mol_name = stem[:-4] if stem.endswith("_opt") else stem
out_dir = os.path.dirname(xyz_path)
os.makedirs(out_dir, exist_ok=True)
# MACEの xyz とは別名。同じ名前に書くと初期構造が消える
opt_xyz_path = os.path.join(out_dir, f"{mol_name}_resp_opt.xyz")

print(f"材料: {mol_name}")
print(f"初期座標 (MACE): {xyz_path}")
print(f"構造最適化: {OPT_XC}/{OPT_BASIS}")
print(f"RESPの単点計算: {SP_XC}/{SP_BASIS}")
print(f"最適化後の座標: {opt_xyz_path}")

材料: TAPC
初期座標 (MACE): output_mace/TAPC/TAPC_opt.xyz
構造最適化: b3lyp/6-31+g(d,p)
RESPの単点計算: hf/6-31g(d)
最適化後の座標: output_mace/TAPC/TAPC_resp_opt.xyz


In [4]:
# MACEの座標から B3LYP/6-31+G(d,p) で構造を取り直す
# {名前}_resp_opt.xyz が既にあるときは、その座標を使い、最適化は繰り返さない
geom_log = os.path.join(out_dir, f"{mol_name}_resp_geomopt.log")

if os.path.exists(opt_xyz_path):
    print(f"既存の最適化構造を使います: {opt_xyz_path}")
else:
    print(f"構造最適化を開始: {OPT_XC}/{OPT_BASIS}")
    print("収束条件は Gaussian の opt=tight に相当する。TAPC は基底関数が約1100個あるので、8GBのGPUでは他のCUDAジョブを止める。")
    mol = gto.M(
        atom=xyz_path,
        basis=OPT_BASIS,
        charge=CHARGE,
        spin=SPIN,
        verbose=4,
        output=geom_log,
    )
    if mol.nelectron % 2 != 0:
        close_pyscf_log(mol)
        raise ValueError(f"電子数が奇数 ({mol.nelectron}) です。このノートブックは閉殻分子用です。")

    mf = rks.RKS(mol, xc=OPT_XC)
    # 交換相関の数値積分。3はPySCFの初期値で、Gaussianの Integral=Fine に近い
    mf.grids.level = 3
    # 密度フィッティングは使わない。記事の Gaussian と同じ4中心積分にする
    # GAU_TIGHT は geomeTRIC にある Gaussian opt=tight の閾値
    conv, mol_opt = geom_kernel(
        mf,
        maxsteps=200,
        convergence_set="GAU_TIGHT",
    )
    if not conv:
        close_pyscf_log(mol)
        raise RuntimeError("構造最適化が200ステップ以内に収束しませんでした。")

    # PySCF内部はBohr。xyzはAngstromで書く（1 Bohr = 0.529177210903 Å）
    opt_coords = mol_opt.atom_coords()
    symbols = [mol_opt.atom_symbol(i) for i in range(mol_opt.natm)]
    xyz_lines = [f"{mol_opt.natm}", f"{mol_name} B3LYP/6-31+G(d,p) opt=tight"]
    for sym, pos in zip(symbols, opt_coords):
        pos_ang = pos * 0.529177210903
        xyz_lines.append(f"{sym} {pos_ang[0]:.6f} {pos_ang[1]:.6f} {pos_ang[2]:.6f}")
    with open(opt_xyz_path, "w") as handle:
        handle.write("\n".join(xyz_lines) + "\n")
    print(f"最適化構造: {opt_xyz_path}")

    close_pyscf_log(mol)
    close_pyscf_log(mol_opt)
    del mf, mol_opt, mol
    release_gpu()

geometric-optimize called with the following command line:
/home/kawano/miniforge3/envs/gpu4pyscf/lib/python3.10/site-packages/ipykernel_launcher.py --f=/mnt/wslg/runtime-dir/jupyter/runtime/kernel-v3b8abc02ed007554227e0e911d406ee49f1735e27.json

                                        ())))))))))))))))/                     
                                    ())))))))))))))))))))))))),                
                                *)))))))))))))))))))))))))))))))))             
                        #,    ()))))))))/                .)))))))))),          
                      #%%%%,  ())))))                        .))))))))*        
                      *%%%%%%,  ))              ..              ,))))))).      
                        *%%%%%%,         ***************/.        .)))))))     
                #%%/      (%%%%%%,    /*********************.       )))))))    
              .%%%%%%#      *%%%%%%,  *******/,     **********,      .))))))   
                .%%%%%%/      *%%

構造最適化を開始: b3lyp/6-31+g(d,p)
収束条件は Gaussian の opt=tight に相当する。TAPC は基底関数が約1100個あるので、8GBのGPUでは他のCUDAジョブを止める。
overwrite output file: output_mace/TAPC/TAPC_resp_geomopt.log


282 internal coordinates being used (instead of 282 Cartesians)
Internal coordinate system (atoms numbered from 1):
Distance 1-2
Distance 1-33
Distance 1-34
Distance 2-3
Distance 2-49
Distance 3-4
Distance 3-50
Distance 4-5
Distance 4-32
Distance 5-6
Distance 5-27
Distance 5-31
Distance 6-7
Distance 6-11
Distance 7-8
Distance 7-51
Distance 8-9
Distance 8-52
Distance 9-10
Distance 9-12
Distance 10-11
Distance 10-53
Distance 11-54
Distance 12-13
Distance 12-20
Distance 13-14
Distance 13-19
Distance 14-15
Distance 14-55
Distance 15-16
Distance 15-56
Distance 16-17
Distance 16-18
Distance 17-57
Distance 17-58
Distance 17-59
Distance 18-19
Distance 18-60
Distance 19-61
Distance 20-21
Distance 20-26
Distance 21-22
Distance 21-62
Distance 22-23
Distance 22-63
Distance 23-24
Distance 23-25
Distance 24-64
Distance 24-65
Distance 24-66
Distance 25-26
Distance 25-67
Distance 26-68
Distance 27-28
Distance 27-69
Distance 27-70
Distance 28-29
Distance 28-71
Distance 28-72
Distance 29-30
Distance 29-

最適化構造: output_mace/TAPC/TAPC_resp_opt.xyz


In [5]:
# 取り直した座標を固定して HF/6-31G(d) の単点計算。この波動関数を RESP に使う
print(f"単点計算を開始: {SP_XC}/{SP_BASIS}")
log_path = os.path.join(out_dir, f"{mol_name}_resp.log")
mol = gto.M(
    atom=opt_xyz_path,
    basis=SP_BASIS,
    charge=CHARGE,
    spin=SPIN,
    verbose=4,
    output=log_path,
)
mf = RHF(mol)
# Hartree-Fockには交換相関の数値積分格子がない。密度フィッティングも使わない
mf.chkfile = os.path.join(out_dir, f"{mol_name}_resp.chk")
mf.kernel()

fchk_path = os.path.join(out_dir, f"{mol_name}_resp.fchk")
write_fchk(mf, fchk_path, title=f"{mol_name} {SP_XC}/{SP_BASIS}")
print(f"E = {float(mf.e_tot):.8f} Hartree")
print(f"fchk: {fchk_path}")
print("このfchkをMultiwfnで開き、RESP電荷を計算する。")

close_pyscf_log(mol)
del mf, mol
release_gpu()

単点計算を開始: hf/6-31g(d)
overwrite output file: output_mace/TAPC/TAPC_resp.log
E = -1877.61513940 Hartree
fchk: output_mace/TAPC/TAPC_resp.fchk
このfchkをMultiwfnで開き、RESP電荷を計算する。
